In [1]:
import pandas as pd
import numpy as np
import os
from datasets import load_dataset
from google.colab import drive
import urllib.request
import zipfile
import requests
import time
from bs4 import BeautifulSoup
import random

In [2]:
!pip install datasets

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# --- STEP 2: Path Settings (Sabse Important) ---
base_dir = '/content/drive/MyDrive/fake news project'
dataset_dir = os.path.join(base_dir, 'data sets')

In [5]:
print(f"Project Folder: {base_dir}")
print(f"Dataset Folder: {dataset_dir}")

Project Folder: /content/drive/MyDrive/fake news project
Dataset Folder: /content/drive/MyDrive/fake news project/data sets


In [6]:
#reading the 1st data sets

fake_path = os.path.join(dataset_dir, "fake_.csv")
true_path = os.path.join(dataset_dir, "True.csv")
print("Files exist?", os.path.exists(fake_path), os.path.exists(true_path))

df_fake = pd.read_csv(fake_path)
df_true = pd.read_csv(true_path)
print("Fake shape:", df_fake.shape, "True shape:", df_true.shape)
display(df_fake.head(2))
display(df_true.head(2))

Files exist? True True


/tmp/ipython-input-4172157238.py:7: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(fake_path)


Fake shape: (23502, 172) True shape: (21417, 4)


,title,text,subject,date,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 162,Unnamed: 163,Unnamed: 164,Unnamed: 165,Unnamed: 166,Unnamed: 167,Unnamed: 168,Unnamed: 169,Unnamed: 170,Unnamed: 171
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"


In [7]:
df_fake.columns

Index(['title', 'text', 'subject', 'date', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 162', 'Unnamed: 163', 'Unnamed: 164', 'Unnamed: 165',
       'Unnamed: 166', 'Unnamed: 167', 'Unnamed: 168', 'Unnamed: 169',
       'Unnamed: 170', 'Unnamed: 171'],
      dtype='object', length=172)

In [8]:
#first dropping the all the column in df_fake except 4
df_fake=df_fake[["title","text","subject","date"]].copy()
df_fake.head()


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [10]:
# Cell 2: label & combine
df_fake['label'] = 1    # fake
df_true['label'] = 0    # real

#keep essential columns: title, text, label
cols_keep = []
for c in ['title','text','label']:
    if c in df_fake.columns:
        cols_keep.append(c)

df_fake = df_fake[cols_keep].copy()
df_true = df_true[cols_keep].copy()

df = pd.concat([df_fake, df_true], ignore_index=True)
print("Combined shape:", df.shape)
display(df.head())

Combined shape: (44919, 3)


,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1


In [11]:
df.label.value_counts()

,count
label,
1,23502
0,21417


## quick cleaning and language filter

In [12]:
# checking how many null values is there
print(f"count of null value")
display(df.isnull().sum())

print(f"count of duplicated row with same title and text")
display(df.duplicated().sum())

count of null value


,0
title,0
text,0
label,0


count of duplicated row with same title and text


np.int64(5803)

In [13]:
# drop duplicates by text
before = len(df)
df.drop_duplicates(subset=['text'], inplace=True)
print(f"Dropped duplicates: {before - len(df)} rows")

Dropped duplicates: 6262 rows


In [14]:
#checking the info of entire data set
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38657 entries, 0 to 44918
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   38657 non-null  object
 1   text    38657 non-null  object
 2   label   38657 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [15]:
print(f"after cleaning the 1st dataset count of true and fake news is:-{df.label.value_counts()}")

after cleaning the 1st dataset count of true and fake news is:-label
0    21191
1    17466
Name: count, dtype: int64


## loading and cleaning the 2nd data sets

In [16]:
#loading the data set name IFND.csv
ifnd_path=os.path.join(dataset_dir,"IFND.csv")
print(ifnd_path)
print(f"path is there? {os.path.exists(ifnd_path)}")
df2=pd.read_csv(ifnd_path,encoding='latin1')
df2.head()

/content/drive/MyDrive/fake news project/data sets/IFND.csv
path is there? True


,id,Statement,Image,Web,Category,Date,Label
0,2,"WHO praises India's Aarogya Setu app, says it ...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE
1,3,"In Delhi, Deputy US Secretary of State Stephen...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,VIOLENCE,Oct-20,TRUE
2,4,LAC tensions: China's strategy behind delibera...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,TERROR,Oct-20,TRUE
3,5,India has signed 250 documents on Space cooper...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE
4,6,Tamil Nadu chief minister's mother passes away...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,ELECTION,Oct-20,TRUE


In [17]:
#checking the no of rows and column in this data set
display(f"no of rows and column:-{(df2.shape)}")
#checking the name of all the column
display(f"column name in datasets:-{df2.columns}")
#checking the duplicate rows and null value count how many are there in a data set
display(f"duplicate rows :-{df2.duplicated(subset="Statement").sum()}")
print(f"null value in dataset:-{df2.isnull().sum()}")


'no of rows and column:-(56714, 7)'

"column name in datasets:-Index(['id', 'Statement', 'Image', 'Web', 'Category', 'Date', 'Label'], dtype='object')"

'duplicate rows :-374'

null value in dataset:-id               0
Statement        0
Image            0
Web              0
Category         0
Date         11321
Label            0
dtype: int64


In [18]:
# Rename columns to match my pipeline
df2.rename(columns={"Statement":"text"},inplace=True)
df2.rename(columns={"Label":"label"},inplace=True)
# Create a simple title
df2["title"]=df2["text"].apply(lambda x: str(x)[:60]) # first 60 chars as title
# now checking whether the column has been rename or new has added or not
print(f"column:-{df2.columns}")

column:-Index(['id', 'text', 'Image', 'Web', 'Category', 'Date', 'label', 'title'], dtype='object')


In [19]:
#checking the unique value in label
print(f"category:-{df2.label.unique()}")
# checking all the column data type first
print(f"info:-{df2.info()}")

category:-['TRUE' 'Fake']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56714 entries, 0 to 56713
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        56714 non-null  int64 
 1   text      56714 non-null  object
 2   Image     56714 non-null  object
 3   Web       56714 non-null  object
 4   Category  56714 non-null  object
 5   Date      45393 non-null  object
 6   label     56714 non-null  object
 7   title     56714 non-null  object
dtypes: int64(1), object(7)
memory usage: 3.5+ MB
info:-None


In [20]:
# we only want the column [ title , text , label ]
# removing all the unwanted column
df2 = df2[['title', 'text', 'label']]
df2.info()
# changing the data type of 3 column
cols = ['title', 'text', 'label']
df[cols] = df[cols].astype('string')
df2.info()
df2.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56714 entries, 0 to 56713
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   56714 non-null  object
 1   text    56714 non-null  object
 2   label   56714 non-null  object
dtypes: object(3)
memory usage: 1.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56714 entries, 0 to 56713
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   56714 non-null  object
 1   text    56714 non-null  object
 2   label   56714 non-null  object
dtypes: object(3)
memory usage: 1.3+ MB


,title,text,label
0,"WHO praises India's Aarogya Setu app, says it ...","WHO praises India's Aarogya Setu app, says it ...",TRUE
1,"In Delhi, Deputy US Secretary of State Stephen...","In Delhi, Deputy US Secretary of State Stephen...",TRUE
2,LAC tensions: China's strategy behind delibera...,LAC tensions: China's strategy behind delibera...,TRUE
3,India has signed 250 documents on Space cooper...,India has signed 250 documents on Space cooper...,TRUE
4,Tamil Nadu chief minister's mother passes away...,Tamil Nadu chief minister's mother passes away...,TRUE
5,Bihar Assembly Election 2020: This is why Tej ...,Bihar Assembly Election 2020: This is why Tej ...,TRUE
6,"Hathras case: CBI reaches victim's village, vi...","Hathras case: CBI reaches victim's village, vi...",TRUE
7,"Rajasthan Crime News: After Karauli, another e...","Rajasthan Crime News: After Karauli, another e...",TRUE
8,"Mumbai: BMC to book, penalise people stepping ...","Mumbai: BMC to book, penalise people stepping ...",TRUE
9,COVID-19: India's single-day spike drops to 55...,COVID-19: India's single-day spike drops to 55...,TRUE


In [21]:
df2.label.value_counts()

,count
label,
TRUE,37800
Fake,18914


In [22]:
# 0 = real (true)
# 1 = fake (false misinformation)

# Standardize labels
fake="Fake"
real="TRUE"
def convertion_label(x):
    if x == fake:
        return 1
    else:
        return 0
df2["label"]=df2["label"].apply(convertion_label)
df2.head()


,title,text,label
0,"WHO praises India's Aarogya Setu app, says it ...","WHO praises India's Aarogya Setu app, says it ...",0
1,"In Delhi, Deputy US Secretary of State Stephen...","In Delhi, Deputy US Secretary of State Stephen...",0
2,LAC tensions: China's strategy behind delibera...,LAC tensions: China's strategy behind delibera...,0
3,India has signed 250 documents on Space cooper...,India has signed 250 documents on Space cooper...,0
4,Tamil Nadu chief minister's mother passes away...,Tamil Nadu chief minister's mother passes away...,0


In [23]:
df2.label.value_counts()


,count
label,
0,37800
1,18914


In [24]:
# removing all the duplicate row (it will check the duplicate only with text column)
df2.drop_duplicates(subset="text",inplace=True)
print(f"after:-{df2.duplicated(subset="text").sum()}")


after:-0


In [25]:

df.label=df.label.astype("int64")
#changing the column datatype for df2
df2[["title","text"]]=df2[["title","text"]].astype("string")
print(f"df_info:-{df.info()}")
print(f"df2_info:-{df2.info()}")

<class 'pandas.core.frame.DataFrame'>
Index: 38657 entries, 0 to 44918
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   38657 non-null  string
 1   text    38657 non-null  string
 2   label   38657 non-null  int64 
dtypes: int64(1), string(2)
memory usage: 1.2 MB
df_info:-None
<class 'pandas.core.frame.DataFrame'>
Index: 56340 entries, 0 to 56713
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   56340 non-null  string
 1   text    56340 non-null  string
 2   label   56340 non-null  int64 
dtypes: int64(1), string(2)
memory usage: 1.7 MB
df2_info:-None


## loading the 3rd datasets

In [26]:
# now loading the datasets
df3_path=os.path.join(dataset_dir,"bharatfakenews.xlsx")

df3=pd.read_excel(df3_path,engine="openpyxl")
df3.head()

,id,Author_Name,Fact_Check_Source,Source_Type,Statement,Eng_Trans_Statement,News Body,Eng_Trans_News_Body,Media_Link,Publish_Date,Fact_Check_Link,News_Category,Language,Region,Platform,Text,Video,Image,Label
0,BFNK_1,Shinjinee Majumder,Alt News,IFCN,फ़ैक्ट-चेक: तेलंगाना में एक रिपोर्टर ने गृह मंत...,Fact-check: A reporter in Telangana stopped sp...,सोशल मीडिया पर एक वीडियो वायरल है जिसमें एक पत...,A video is viral on social media in which a jo...,https://i0.wp.com/www.altnews.in/Hindi/wp-cont...,9th July 2022,https://www.altnews.in/Hindi/fact-check-was-am...,Politics,Hindi,Telangana,Twitter,no,yes,no,False
1,BFNK_2,Kalim Ahmed,Alt News,IFCN,PM मोदी को UAE का सर्वोच्च नागरिक सम्मान मिलने...,Share by stating the old video of PM Modi's hi...,प्रधानमंत्री नरेंद्र मोदी को सोने की चेन से सम...,A video of Prime Minister Narendra Modi being ...,https://i0.wp.com/www.altnews.in/Hindi/wp-cont...,9th July 2022,https://www.altnews.in/Hindi/old-video-of-pm-m...,Politics,Hindi,National,Twitter,no,yes,no,False
2,BFNK_3,Abhishek Kumar,Alt News,IFCN,वायरल तस्वीर में सुप्रीम कोर्ट के जज सूर्यकांत...,Supreme Court judges Suryakant and JB Pardiwal...,बीते दिनों नूपुर शर्मा ने टीवी डिबेट में पैगम्...,"Recently, Nupur Sharma made an objectionable c...",https://i0.wp.com/www.altnews.in/Hindi/wp-cont...,7th July 2022,https://www.altnews.in/Hindi/false-claim-with-...,Politics,Hindi,National,Twitter,no,no,yes,False
3,BFNK_4,Abhishek Kumar,Alt News,IFCN,मीडिया ने दी ग़लत ख़बर: कटनी में मुस्लिम सरपंच क...,Media gave wrong news: After the victory of Mu...,एक वीडियो सोशल मीडिया पर वायरल है. इसे शेयर कर...,A video is viral on social media. While sharin...,https://i0.wp.com/www.altnews.in/Hindi/wp-cont...,5th July 2022,https://www.altnews.in/Hindi/media-misreport-p...,Politics,Hindi,Madhya Pradesh,Twitter,no,no,yes,False
4,BFNK_5,Kinjal,Alt News,IFCN,महिला ने राहुल गांधी को कश्मीर मुद्दे पर मोदी ...,The woman lashed out at Rahul Gandhi to oppose...,सोशल मीडिया पर राहुल गांधी का एक वीडियो वायरल ...,A video of Rahul Gandhi has gone viral on soci...,https://i0.wp.com/www.altnews.in/Hindi/wp-cont...,4th July 2022,https://www.altnews.in/Hindi/2019-video-of-wom...,Politics,Hindi,Kashmir,Twitter,no,yes,no,True


In [27]:
#first reading the column
print(f"column:-{df3.columns}")
#now taking the useful column only
columns=["Eng_Trans_Statement","Eng_Trans_News_Body","Label"]
df3=df3[columns]

column:-Index(['id', 'Author_Name', 'Fact_Check_Source', 'Source_Type', 'Statement',
       'Eng_Trans_Statement', 'News Body', 'Eng_Trans_News_Body', 'Media_Link',
       'Publish_Date', 'Fact_Check_Link', 'News_Category', 'Language',
       'Region', 'Platform', 'Text', 'Video', 'Image', 'Label'],
      dtype='object')


In [28]:
# now checking the info of the df3 sets
print(f"info:-{df3.info()}")
print("\n")

#first changing the column name
df3.rename(columns={"Eng_Trans_Statement":"title",
                    "Eng_Trans_News_Body":"text",
                    "Label":"label"},inplace=True)
#first removing the duplicate row and null value
print(f"null_value_in_df3:-{df3.isnull().sum()}")
print("\n")
print(f"duplicated_row_in_df3:-{df3.duplicated(subset="text").sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26232 entries, 0 to 26231
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Eng_Trans_Statement  26232 non-null  object
 1   Eng_Trans_News_Body  25360 non-null  object
 2   Label                26232 non-null  bool  
dtypes: bool(1), object(2)
memory usage: 435.6+ KB
info:-None


null_value_in_df3:-title      0
text     872
label      0
dtype: int64


duplicated_row_in_df3:-1864


In [29]:
#now removing all the duplicate row
df3.drop_duplicates(subset="text",inplace=True)
df3.dropna(subset="text",inplace=True)
print(f"rows and column:-{df3.shape}")


rows and column:-(24367, 3)


In [30]:
# 0 = real (true)
# 1 = fake (false misinformation)

#first checking the no of category in label
print(f"label_Category:-{df3.label.unique()}")

label_Category:-[False  True]


In [31]:
df3.label.value_counts()

,count
label,
True,14818
False,9549


In [32]:
#Standardize labels
fake=False
real=True
def convertion_label(x):
    if x == fake:
        return 1
    else:
        return 0
df3["label"]=df3["label"].apply(convertion_label)
df3.head()

,title,text,label
0,Fact-check: A reporter in Telangana stopped sp...,A video is viral on social media in which a jo...,1
1,Share by stating the old video of PM Modi's hi...,A video of Prime Minister Narendra Modi being ...,1
2,Supreme Court judges Suryakant and JB Pardiwal...,"Recently, Nupur Sharma made an objectionable c...",1
3,Media gave wrong news: After the victory of Mu...,A video is viral on social media. While sharin...,1
4,The woman lashed out at Rahul Gandhi to oppose...,A video of Rahul Gandhi has gone viral on soci...,0


In [33]:
df3.label.value_counts()

,count
label,
0,14818
1,9549


## laoding the 4th datasets


In [34]:
df4_path=os.path.join(dataset_dir,"fake.csv")
df4=pd.read_csv(df4_path)

In [35]:
#checking the no of rows and column
print(f"rows and column df4:-{df4.shape}")
print("")
#checking the column name
print(f"column:-{df4.columns}")

rows and column df4:-(12999, 20)

column:-Index(['uuid', 'ord_in_thread', 'author', 'published', 'title', 'text',
       'language', 'crawled', 'site_url', 'country', 'domain_rank',
       'thread_title', 'spam_score', 'main_img_url', 'replies_count',
       'participants_count', 'likes', 'comments', 'shares', 'type'],
      dtype='object')


In [36]:
# checking the label category
df4.type.unique()

array(['bias', 'conspiracy', 'fake', 'bs', 'satire', 'hate', 'junksci',
       'state'], dtype=object)

In [37]:
fake_label=["bias","conspiracy","fake","bs","satire","hate","junksci","state"]
#real-0
#fake-1
# now just labelling the value in number
def convertion_label(x):
    if x in fake_label:
        return 1
    else:
        return 0
df4.type=df4.type.apply(convertion_label)

In [38]:
df4.columns

Index(['uuid', 'ord_in_thread', 'author', 'published', 'title', 'text',
       'language', 'crawled', 'site_url', 'country', 'domain_rank',
       'thread_title', 'spam_score', 'main_img_url', 'replies_count',
       'participants_count', 'likes', 'comments', 'shares', 'type'],
      dtype='object')

In [39]:
#renaming the column name
column_change_name={
    "type":"label"
}
df4.rename(columns=column_change_name,inplace=True)

In [40]:
#selecting useful column
useful_column=["title","text","label"]
df4=df4[useful_column]
print(f"shape:-{df4.shape}")

shape:-(12999, 3)


In [41]:
# droping the null value
print(f"before:-{df4.isnull().sum()}")
print("")
df4.dropna(inplace=True)
print(f"after:-{df4.isnull().sum()}")

before:-title    680
text      46
label      0
dtype: int64

after:-title    0
text     0
label    0
dtype: int64


In [42]:
#cheking the duplicate value
print(f"before:-{df4.duplicated(subset="text").sum()}")
df4.drop_duplicates(subset="text",inplace=True)
print(f"after:-{df4.duplicated(subset="text").sum()}")

before:-472
after:-0


In [43]:
df4.label.value_counts()

,count
label,
1,11801


In [44]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11801 entries, 0 to 12912
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   11801 non-null  object
 1   text    11801 non-null  object
 2   label   11801 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 368.8+ KB


In [45]:
#changing the type
df4[["title","text"]]=df4[["title","text"]].astype("string")

In [46]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11801 entries, 0 to 12912
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   11801 non-null  string
 1   text    11801 non-null  string
 2   label   11801 non-null  int64 
dtypes: int64(1), string(2)
memory usage: 368.8 KB


In [47]:
df4.head()

,title,text,label
0,Muslims BUSTED: They Stole Millions In Gov’t B...,Print They should pay all the back all the mon...,1
1,Re: Why Did Attorney General Loretta Lynch Ple...,Why Did Attorney General Loretta Lynch Plead T...,1
2,BREAKING: Weiner Cooperating With FBI On Hilla...,Red State : Fox News Sunday reported this mor...,1
3,PIN DROP SPEECH BY FATHER OF DAUGHTER Kidnappe...,Email Kayla Mueller was a prisoner and torture...,1
4,FANTASTIC! TRUMP'S 7 POINT PLAN To Reform Heal...,Email HEALTHCARE REFORM TO MAKE AMERICA GREAT ...,1


## laoding the 5th dataset


In [48]:
df5_path=os.path.join(dataset_dir,"WELFake_Dataset.csv")
df5=pd.read_csv(df5_path)

In [49]:
# now checking the column
print(f"column:-{df5.columns}")
# checking the no of rows and columns
print(f"rows and column:-{df5.shape}")

column:-Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')
rows and column:-(72134, 4)


In [50]:
# now selecting only the 3 column
df5=df5[["title","text","label"]]


In [51]:
# now checking hte label unique value
print(f"label:-{df5.label.unique()}")
# now printing the count of true news and false news (true:1 and false:0)
print(f"count:-{df5.label.value_counts()}")
#

label:-[1 0]
count:-label
1    37106
0    35028
Name: count, dtype: int64


In [52]:
# changing the label 0 as true and 1 as fake
def convertion_label(x):
  if x==1:
    return 0
  else:
    return 1

df5.label=df5.label.apply(convertion_label)

In [53]:
# now checking the count
print(f"count:-{df5.label.value_counts()}")

count:-label
0    37106
1    35028
Name: count, dtype: int64


In [54]:
# now checking hte null value
print(f"null value in datasets:-{df5.isnull().sum()}")

null value in datasets:-title    558
text      39
label      0
dtype: int64


In [55]:
df5.dropna(inplace=True)

In [56]:
# checking the count after removing it
print(f"after removing null value in datasets:-{df5.isnull().sum()}")

after removing null value in datasets:-title    0
text     0
label    0
dtype: int64


In [57]:
# now checking the duplicate value as all
print(f"duplicate for entire row:-{df5.duplicated().sum()}")

# now checking the duplicate row only text
print(f"duplicate for text column:-{df5.duplicated(subset="text").sum()}")

duplicate for entire row:-8416
duplicate for text column:-9337


In [58]:
# removing all the duplicate text
df5.drop_duplicates(subset="text",inplace=True)
# now checking the count after removing duplicate text row

In [59]:
df5.duplicated(subset="text").sum()

np.int64(0)

In [60]:
# now checking the count of total true and fake news
print(f"final_count:_{df5.label.value_counts()}")

final_count:_label
1    34620
0    27580
Name: count, dtype: int64


In [61]:
df5.head(5)

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,0
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",0
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,1
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",0
5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,0


## now downloading the 6th dataet name LIAR from huggingface

In [62]:
# Path set kar (SPACE ko dhyan se likha hai!)
save_path = '/content/drive/MyDrive/fake news project/data sets/'
os.makedirs(save_path, exist_ok=True)

print("LIAR Dataset download ho raha hai (Direct source se)...")

# --- DIRECT UCSB SOURCE SE DOWNLOAD ---
url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
zip_path = os.path.join(save_path, 'liar_dataset.zip')

try:
    print(f"Downloading from: {url}")
    urllib.request.urlretrieve(url, zip_path)
    print("✅ Download complete!")

    # Extract kar
    print("\nExtracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(save_path)
    print("✅ Extraction complete!")

    # Check extracted folder structure
    extracted_files = os.listdir(save_path)
    print(f"\nExtracted files/folders: {extracted_files}")

    # Train.tsv dhundo (folder structure dekh kar)
    import glob
    train_files = glob.glob(os.path.join(save_path, '**', 'train.tsv'), recursive=True)

    if not train_files:
        print("❌ train.tsv nahi mila!")
        print(f"All files: {os.listdir(save_path)}")
        raise FileNotFoundError("train.tsv not found in extracted files")

    train_file = train_files[0]
    print(f"\nFound train file at: {train_file}")

    # LIAR format: tab-separated, no header
    df_liar = pd.read_csv(
        train_file,
        sep='\t',
        header=None,
        names=['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job',
               'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true',
               'pants_fire', 'context']
    )

    print(f"✅ Data loaded! Shape: {df_liar.shape}")
    print(f"\nColumns: {df_liar.columns.tolist()}")
    print(f"\nFirst 3 rows:")
    print(df_liar.head(3))

except Exception as e:
    print(f"❌ Error: {e}")
    print("\nManual download try kar: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip")
    raise


LIAR Dataset download ho raha hai (Direct source se)...
✅ Download complete!

Extracting files...
✅ Extraction complete!

Extracted files/folders: ['IFND.csv', 'fake.csv', 'True.csv', 'fake_.csv', 'bharatfakenews.xlsx', 'WELFake_Dataset.csv', 'liar_dataset.zip', 'test.tsv', 'train.tsv', 'valid.tsv', 'README', 'Final_HuggingFace_LIAR.csv', 'API_Real_News_Large.csv', 'Scraped_Fake_News.csv']

Found train file at: /content/drive/MyDrive/fake news project/data sets/train.tsv
✅ Data loaded! Shape: (10240, 14)

Columns: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true', 'pants_fire', 'context']

First 3 rows:
           id        label                                          statement  \
0   2635.json        false  Says the Annies List political group supports ...   
1  10540.json    half-true  When did the decline of coal start? It started...   
2    324.json  mostly-true  Hillary Clinton agrees with John 

In [63]:
# ============ DATA CLEANING ============

print("\n" + "="*50)
print("Data Processing...")

# Keep sirf statement aur label
df_liar = df_liar[['statement', 'label']].copy()

# Label mapping
# LIAR labels: true, mostly-true, half-true, barely-true, false, pants-fire
# Mapping: true/mostly-true -> 1 (REAL), false/pants-fire -> 0 (FAKE)

label_mapping = {
    'true': 0,
    'mostly-true': 0,
    'half-true': 1,  # Ambiguous, lekin false side mein le lo
    'barely-true': 1,
    'false': 1,
    'pants-fire': 1
}

df_liar['binary_label'] = df_liar['label'].map(label_mapping)


Data Processing...


In [ ]:
df_liar.head()

,statement,label,binary_label
0,Says the Annies List political group supports ...,false,1
1,When did the decline of coal start? It started...,half-true,1
2,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,0
3,Health care reform legislation is likely to ma...,false,1
4,The economic turnaround started at the end of ...,half-true,1


In [64]:
print(f"count of true and false:-{df_liar.binary_label.value_counts()}")
print("")
print(f"no of null value in the liar datasets:-{df_liar.isnull().sum()}")

count of true and false:-binary_label
1    6602
0    3638
Name: count, dtype: int64

no of null value in the liar datasets:-statement       0
label           0
binary_label    0
dtype: int64


In [65]:
print(f"information of all the liar dataset:-{df_liar.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10240 entries, 0 to 10239
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   statement     10240 non-null  object
 1   label         10240 non-null  object
 2   binary_label  10240 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 240.1+ KB
information of all the liar dataset:-None


In [66]:
# now selecting only the column statement and bineary labels
df_liar=df_liar[["statement","binary_label"]]
#now renaming the column
df_liar.rename(columns={"statement":"text","binary_label":"label"},inplace=True)

In [67]:
# now checking the duplicate text
print(f"no of duplicate text:-{df_liar.duplicated(subset="text").sum()}")
# now droping the duplicate text row
df_liar.drop_duplicates(subset="text",inplace=True)
# after that
print(f" after removing no of duplicate text:-{df_liar.duplicated(subset="text").sum()}")

no of duplicate text:-17
 after removing no of duplicate text:-0


In [68]:
# now making the extra column name title and content in both the column there will be same content text
df_liar["title"]=df_liar.text
df_liar["content"]=df_liar.text

In [69]:
# now saving the datasets in the folder dataset
# Save file
output_file = os.path.join(save_path, 'Final_HuggingFace_LIAR.csv')
df_liar.to_csv(output_file, index=False)
print(f"\n✅ SUCCESS! File saved at:")


✅ SUCCESS! File saved at:


## merging all the datasets together

In [70]:
final_dataset=pd.concat([df,df2,df3,df4,df5],ignore_index=True)

In [71]:
# now making the new column name content by adding the title and text onces
final_dataset["content"]=final_dataset["title"]+" "+final_dataset["text"]


In [72]:
final_dataset=pd.concat([final_dataset,df_liar],ignore_index=True)

In [73]:
# now checking the no of row and no of column
print(f"no of row and column:-{final_dataset.shape}")


no of row and column:-(203588, 4)


In [74]:
# checking the count of true and false
print(f"count of true and false:-{final_dataset.label.value_counts()}")
#

count of true and false:-label
0    105024
1     98564
Name: count, dtype: int64


In [75]:
final_dataset.content.head(1)


,content
0,Donald Trump Sends Out Embarrassing New Year’...


In [78]:
# now saving this final dataset into the new folder under fake news project folder name will be cleaned kaggle datasets
cleaned_dataset_path=os.path.join(base_dir,"V2_cleaned kaggle datasets")
os.makedirs(cleaned_dataset_path,exist_ok=True)


In [79]:
final_dataset.drop_duplicates(subset="content",inplace=True)

In [80]:
final_dataset.isnull().sum()

,0
title,0
text,0
label,0
content,0


In [81]:
final_dataset.dropna(subset="content",inplace=True)
final_dataset.duplicated(subset="content").sum()

np.int64(0)

In [82]:
# now checking the null value
print(f"after cleaning null value:-{final_dataset.isnull().sum()}")

after cleaning null value:-title      0
text       0
label      0
content    0
dtype: int64


In [83]:
print(f"after cleaning shape of the dataset is:-{final_dataset.shape}")

print(f"after cleaning checking the count of true and false news:-{final_dataset.label.value_counts()}")


after cleaning shape of the dataset is:-(154935, 4)
after cleaning checking the count of true and false news:-label
0    77558
1    77377
Name: count, dtype: int64


## now taking the datasets fron API

In [84]:
# --- CONFIGURATION ---
api_key = '2143fcae8a6b48ebb3c05d1dc25f16e6'  # Teri Key
base_url = "https://newsapi.org/v2/everything"

# --- MASSIVE KEYWORD LIST (Har field cover karenge) ---
# Hum ~80 keywords use kar rahe hain.
# 80 keywords * 100 articles = ~8000 articles (Duplicates hatne ke baad ~6-7k rahenge)
keywords = [
    # World & Politics
    'united nations', 'g20 summit', 'white house', 'parliament india', 'ukraine war',
    'israel gaza', 'nato alliance', 'european union', 'geopolitics', 'democracy',
    'elections 2024', 'human rights', 'supreme court', 'prime minister', 'president',

    # Business & Economy
    'stock market', 'inflation', 'recession', 'cryptocurrency', 'bitcoin',
    'ethereum', 'startup india', 'silicon valley', 'gold price', 'oil prices',
    'real estate', 'banking sector', 'taxes', 'trade war', 'global economy',

    # Technology & Science
    'artificial intelligence', 'machine learning', 'chatgpt', 'google gemini', 'cyber security',
    'hacking', 'data science', 'nasa space', 'isro india', 'mars mission',
    'electric vehicles', 'tesla', 'iphone 16', '5g network', 'metaverse',
    'robotics', 'climate change', 'renewable energy', 'solar power', 'quantum computing',

    # Sports
    'cricket world cup', 'ipl cricket', 'fifa football', 'premier league', 'nba basketball',
    'tennis grand slam', 'formula 1 racing', 'olympics', 'virat kohli', 'lionel messi',

    # Entertainment & Lifestyle
    'bollywood news', 'hollywood movies', 'netflix series', 'music concert', 'celebrity gossip',
    'video games', 'playstation', 'fashion trends', 'travel guide', 'food recipes',

    # Health & Society
    'mental health', 'covid-19 variant', 'cancer research', 'yoga wellness', 'nutrition diet',
    'education system', 'crime news', 'social media trends', 'pollution control'
]

news_list = []

print(f"Targeting {len(keywords)} topics to get max data... 🚀")
print("Bhai thoda wait karna, ye lamba chalega...")

for i, topic in enumerate(keywords):
    # SortBy 'publishedAt' taaki latest mile, 'pageSize=100' max limit hai
    url = f"{base_url}?q={topic}&language=en&sortBy=publishedAt&pageSize=100&apiKey={api_key}"

    try:
        response = requests.get(url)
        data = response.json()

        if data['status'] == 'ok':
            articles = data['articles']
            count = len(articles)
            print(f"[{i+1}/{len(keywords)}] Topic: '{topic}' -> {count} news mili.")

            for article in articles:
                title = article['title'] if article['title'] else ""
                desc = article['description'] if article['description'] else ""

                if title and desc:
                    content = title + " " + desc
                    # Hum topic bhi save kar rahe hain taaki pata rahe kahan se aaya
                    news_list.append([title, desc, content, 1]) # 1 = Real News

        elif data['status'] == 'error' and data['code'] == 'rateLimited':
            print("\n⚠️ ALERT: API Limit Khatam ho gayi hai!")
            print("Jitna data mila hai usko save kar rahe hain...")
            break # Loop rok do

        else:
            print(f"Skipping {topic}: {data.get('message')}")

    except Exception as e:
        print(f"Error in {topic}: {e}")

    # API ko thoda saans lene do (Safe side)
    time.sleep(0.5)

# --- PROCESSING & SAVING ---
if len(news_list) > 0:
    df_real = pd.DataFrame(news_list, columns=['title', 'text', 'content', 'label'])

    print("-" * 30)
    print(f"Raw Data Collected: {len(df_real)}")

    # 1. Duplicates Udaao (Bahot zaroori hai kyunki 'Bitcoin' aur 'Crypto' me same news ho sakti hai)
    df_real.drop_duplicates(subset=['content'], inplace=True)
    print(f"Unique Data after cleaning: {len(df_real)}")

    # 2. Shuffle
    df_real = df_real.sample(frac=1, random_state=42).reset_index(drop=True)

else:
    print("Kuch bhi data nahi mila bhai. API Key check kar ya kal try karna.")

Targeting 79 topics to get max data... 🚀
Bhai thoda wait karna, ye lamba chalega...
[1/79] Topic: 'united nations' -> 91 news mili.
[2/79] Topic: 'g20 summit' -> 83 news mili.
[3/79] Topic: 'white house' -> 88 news mili.
[4/79] Topic: 'parliament india' -> 97 news mili.
[5/79] Topic: 'ukraine war' -> 90 news mili.
[6/79] Topic: 'israel gaza' -> 91 news mili.
[7/79] Topic: 'nato alliance' -> 89 news mili.
[8/79] Topic: 'european union' -> 100 news mili.
[9/79] Topic: 'geopolitics' -> 96 news mili.
[10/79] Topic: 'democracy' -> 94 news mili.
[11/79] Topic: 'elections 2024' -> 92 news mili.
[12/79] Topic: 'human rights' -> 86 news mili.
[13/79] Topic: 'supreme court' -> 96 news mili.
[14/79] Topic: 'prime minister' -> 87 news mili.
[15/79] Topic: 'president' -> 89 news mili.
[16/79] Topic: 'stock market' -> 98 news mili.
[17/79] Topic: 'inflation' -> 96 news mili.
[18/79] Topic: 'recession' -> 98 news mili.
[19/79] Topic: 'cryptocurrency' -> 100 news mili.
[20/79] Topic: 'bitcoin' -> 99 n

In [85]:
# in this all the label is true but it should be represent as 0
df_real.label=0

In [86]:
#checking the changes
df_real.label.value_counts()

,count
label,
0,5764


In [87]:
 # 3. Saving the dataset in the datasets folder
save_path = '/content/drive/MyDrive/fake news project/data sets/V2_API_Real_News_Large.csv'

# Folder check
folder_path = '/content/drive/MyDrive/fake news project/data sets/'
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

df_real.to_csv(save_path, index=False)

print(f"\n✅ SUCCESS! File saved at: {save_path}")
print(df_real.head())


✅ SUCCESS! File saved at: /content/drive/MyDrive/fake news project/data sets/V2_API_Real_News_Large.csv
                                               title  \
0  US to send ICE agents to Winter Olympics, prom...   
1  Austin honors Marcia Ball, whose nonprofit HOM...   
2  Deep learning models to map osteocyte networks...   
3             Mikel: The reaction has been excellent   
4  Miley Cyrus Earns Her First Hit On One Radio C...   

                                                text  \
0  US to send ICE agents to Winter Olympics, prom...   
1  A tribute concert at the Paramount celebrates ...   
2  Author summary Osteocytes make up the vast maj...   
3  Mikel Arteta has been delighted with the respo...   
4  Miley Cyrus earns a career first win as “Know ...   

                                             content  label  
0  US to send ICE agents to Winter Olympics, prom...      0  
1  Austin honors Marcia Ball, whose nonprofit HOM...      0  
2  Deep learning models to map oste

## now scraping the datasets from the website


In [88]:
# --- CONFIGURATION ---
# Target: 4000 News
# 250 Pages * 20 News per page = ~5000 News (Fake + True mix)
# Filter karne ke baad approx 3500-4000 Fake mil jayengi.
pages_to_scrape = 250

fake_news_list = []

print(f"Mission 4000: PolitiFact se {pages_to_scrape} pages scrape kar rahe hain... ⏳")
print("Isme 10-15 minute lagenge. Please wait...")

base_url = "https://www.politifact.com/factchecks/list/?page="

for page_num in range(1, pages_to_scrape + 1):
    url = base_url + str(page_num)

    try:
        response = requests.get(url)

        # Agar server ne block kiya (status code 200 nahi hai), toh ruk jao
        if response.status_code != 200:
            print(f"⚠️ Page {page_num} par block ho gaye or error aaya. Saving current data...")
            break

        soup = BeautifulSoup(response.text, 'html.parser')

        # News items dhundo
        items = soup.find_all('li', class_='o-listicle__item')

        if not items:
            print(f"Page {page_num} khali hai. Process stopping.")
            break

        for item in items:
            try:
                # 1. Statement
                quote_div = item.find('div', class_='m-statement__quote')
                if not quote_div: continue
                text = quote_div.find('a').text.strip()

                # 2. Verdict
                meter_div = item.find('div', class_='m-statement__meter')
                if not meter_div: continue
                img = meter_div.find('img')
                if not img: continue
                verdict = img['alt'].lower()

                # 3. Filter: Sirf 'False', 'Fake', 'Pants on Fire'
                if 'false' in verdict or 'fire' in verdict or 'fake' in verdict:
                    # Content, content, content, label format taaki main data se match kare
                    fake_news_list.append([text, text, text, 0]) # 0 = Fake
            except:
                continue

        # Progress Update (Har 10 page pe batayega)
        if page_num % 10 == 0:
            print(f"--> Page {page_num} Done. Abhi tak {len(fake_news_list)} fake news mili.")

        # Random Delay (Important taaki IP block na ho)
        # 0.5 se 1.5 second ke beech rukega
        time.sleep(random.uniform(0.5, 1.5))

    except Exception as e:
        print(f"Error on Page {page_num}: {e}")
        # Error aane par bhi loop chalne do, shayad agla page chal jaye

# --- SAVE DATA ---
if len(fake_news_list) > 0:
    df_fake = pd.DataFrame(fake_news_list, columns=['title', 'text', 'content', 'label'])

    # Duplicates hatana zaroori hai
    df_fake.drop_duplicates(subset=['content'], inplace=True)



Mission 4000: PolitiFact se 250 pages scrape kar rahe hain... ⏳
Isme 10-15 minute lagenge. Please wait...
--> Page 10 Done. Abhi tak 204 fake news mili.
--> Page 20 Done. Abhi tak 467 fake news mili.
--> Page 30 Done. Abhi tak 754 fake news mili.
--> Page 40 Done. Abhi tak 1007 fake news mili.
--> Page 50 Done. Abhi tak 1253 fake news mili.
--> Page 60 Done. Abhi tak 1518 fake news mili.
--> Page 70 Done. Abhi tak 1768 fake news mili.
--> Page 80 Done. Abhi tak 2004 fake news mili.
--> Page 90 Done. Abhi tak 2236 fake news mili.
--> Page 100 Done. Abhi tak 2492 fake news mili.
--> Page 110 Done. Abhi tak 2750 fake news mili.
--> Page 120 Done. Abhi tak 2971 fake news mili.
--> Page 130 Done. Abhi tak 3205 fake news mili.
--> Page 140 Done. Abhi tak 3438 fake news mili.
--> Page 150 Done. Abhi tak 3644 fake news mili.
--> Page 160 Done. Abhi tak 3865 fake news mili.
--> Page 170 Done. Abhi tak 4089 fake news mili.
--> Page 180 Done. Abhi tak 4297 fake news mili.
--> Page 190 Done. Abhi 

In [89]:
# checking the first 5 rows of fake news
df_fake.head()

,title,text,content,label
0,"""Right now we're focused on Minneapolis becaus...","""Right now we're focused on Minneapolis becaus...","""Right now we're focused on Minneapolis becaus...",0
1,"An image shows Alex Pretti holding a gun, not ...","An image shows Alex Pretti holding a gun, not ...","An image shows Alex Pretti holding a gun, not ...",0
2,Protesters against the federal immigration cra...,Protesters against the federal immigration cra...,Protesters against the federal immigration cra...,0
3,Evan Kilgore is one of the Border Patrol agent...,Evan Kilgore is one of the Border Patrol agent...,Evan Kilgore is one of the Border Patrol agent...,0
4,"Donald Trump posted on Truth Social, “Only cri...","Donald Trump posted on Truth Social, “Only cri...","Donald Trump posted on Truth Social, “Only cri...",0


In [90]:
# checking whether the content is duplicate of null is there or not
print(f"null value in the datasets:-{df_fake.isnull().sum()}")
print(" ")
print(f"duplicate value in the datasets:-{df_fake.duplicated(subset="content").sum()}")


null value in the datasets:-title      0
text       0
content    0
label      0
dtype: int64
 
duplicate value in the datasets:-0


In [91]:
# now changing the label 0 to 1  because in all the dataset there is 1 for fake
df_fake.label=1

In [92]:
# now saving the datasets into data sets folders
# Path set kar
save_path = '/content/drive/MyDrive/fake news project/data sets/V2_Scraped_Fake_News.csv'

# Folder check
folder_path = '/content/drive/MyDrive/fake news project/data sets/'
if not os.path.exists(folder_path):
  os.makedirs(folder_path)

df_fake.to_csv(save_path, index=False)

print("-" * 30)
print(f"✅ MISSION COMPLETE! 🎯")
print(f"Total Unique Fake News Saved: {len(df_fake)}")
print(df_fake.head())

------------------------------
✅ MISSION COMPLETE! 🎯
Total Unique Fake News Saved: 5719
                                               title  \
0  "Right now we're focused on Minneapolis becaus...   
1  An image shows Alex Pretti holding a gun, not ...   
2  Protesters against the federal immigration cra...   
3  Evan Kilgore is one of the Border Patrol agent...   
4  Donald Trump posted on Truth Social, “Only cri...   

                                                text  \
0  "Right now we're focused on Minneapolis becaus...   
1  An image shows Alex Pretti holding a gun, not ...   
2  Protesters against the federal immigration cra...   
3  Evan Kilgore is one of the Border Patrol agent...   
4  Donald Trump posted on Truth Social, “Only cri...   

                                             content  label  
0  "Right now we're focused on Minneapolis becaus...      1  
1  An image shows Alex Pretti holding a gun, not ...      1  
2  Protesters against the federal immigration cra...

## now merging the data of both scraped one and api ones


In [93]:
final_dataset=pd.concat([final_dataset,df_fake,df_real],ignore_index=True)

In [94]:
print(f"checking the total no of rows:-{final_dataset.shape}")
print(" ")
print(f"before checking the duplicate value in term of content:-{final_dataset.duplicated(subset="content").sum()}")
print(" ")
print(f"before checking the null value in the datasets:-{final_dataset.isnull().sum()}")
print(" ")
print(f"checking the count of true and false news:-{final_dataset.label.value_counts()}")

checking the total no of rows:-(166418, 4)
 
before checking the duplicate value in term of content:-0
 
before checking the null value in the datasets:-title      0
text       0
label      0
content    0
dtype: int64
 
checking the count of true and false news:-label
0    83322
1    83096
Name: count, dtype: int64


In [96]:
final_dataset.to_csv(os.path.join(cleaned_dataset_path,
    "final_dataset.csv"), index=False)

In [97]:
# now checking the info of the final datasets
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166418 entries, 0 to 166417
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   title    166418 non-null  object
 1   text     166418 non-null  object
 2   label    166418 non-null  int64 
 3   content  166418 non-null  object
dtypes: int64(1), object(3)
memory usage: 5.1+ MB
